In [1]:
!pip install vllm pyngrok

In [1]:
import os
from pyngrok import ngrok

# Укажи свой токен ngrok (получи бесплатно на ngrok.com)
# ngrok.set_auth_token("3LhgRvharlRVSBCbXM6SWudOa9I_5nSkcAiAeezbxwzAZsNFp")

import time
time.sleep(30) # Ждем, пока модель загрузится в VRAM

# # Пробрасываем локальный порт 8000 в интернет
# public_url = ngrok.connect(8000)
# print(f"Твой LLM Endpoint: {public_url}/v1")

In [16]:
!pkill -f vllm
!pkill -f ray

In [ ]:
# Запускаем vLLM в фоне.
# max-model-len=8192 критически важен, чтобы не вылететь из памяти GPU!
!vllm serve axteam/Qwen3.5-4B-AWQ-4bit \
    --host 0.0.0.0 \
    --port 8000 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.92 \
    --trust-remote-code


W1105 22:38:34.358000 35464 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1105 22:38:34.359000 35464 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1105 22:38:34.359000 35464 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(APIServer pid=35464) INFO 11-05 22:38:45 [startup.py:38] 
(APIServer pid=35464) INFO 11-05 22:38:45 [startup.py:38]        █     █     █▄   ▄█
(APIServer pid=35464) INFO 11-05 22:38:45 [startup.py:38]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.33.0
(APIServer pid=35464) INFO 11-05 22:38:45 [startup.py:38]   █▄█▀ █     █     █     █  model   axteam/Qwen3.5-4B-AWQ-4bit
(APIServer pid=35464) INFO 11-05 22:38:45 [startup.py:38]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=35464) INFO 11-05 22:38:45 [startup.py:38] 
(APIServer pid=35464) INFO 11-05 22:38:45 [startup.py:96] non-default args: {'model_tag': 'axteam/Qwen3.5-4B-AWQ

In [ ]:
!tail -n 50 vllm.log

In [ ]:
!pip install rank_bm25

In [ ]:
# rag_system/indexer.py
import json
import pickle
import numpy as np
from pathlib import Path
from typing import List, Dict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import torch

class CorpusIndexer:
    def __init__(self, embedding_model_name: str = "BAAI/bge-m3"):
        self.embedding_model_name = embedding_model_name
        self.embedding_model = None
        self.bm25 = None
        self.corpus_blocks = []  # Список всех блоков
        self.block_id_to_idx = {}  # Маппинг block_id -> индекс
        self.dense_vectors = None

    def load_corpus(self, corpus_path: str):
        """Загружает корпус из JSONL файла"""
        self.corpus_blocks = []
        with open(corpus_path, 'r', encoding='utf-8') as f:
            for line in f:
                block = json.loads(line.strip())
                self.corpus_blocks.append(block)

        self.block_id_to_idx = {b['block_id']: i for i, b in enumerate(self.corpus_blocks)}
        print(f"Loaded {len(self.corpus_blocks)} blocks")

    def _get_texts_for_indexing(self) -> List[str]:
        """Подготавливает тексты для индексации"""
        texts = []
        for block in self.corpus_blocks:
            # Можно добавить секцию для контекста
            text = f"{block.get('section', '')} {block['text']}".strip()
            texts.append(text)
        return texts

    def build_bm25_index(self):
        """Строит BM25 индекс"""
        texts = self._get_texts_for_indexing()
        # Токенизация для русского языка (простая по пробелам + пунктуация)
        tokenized = [text.lower().split() for text in texts]
        self.bm25 = BM25Okapi(tokenized)
        print("BM25 index built")

    def build_dense_index(self, batch_size: int = 32):
        """Строит Dense индекс с использованием embedding модели"""
        if self.embedding_model is None:
            print(f"Loading embedding model: {self.embedding_model_name}")
            self.embedding_model = SentenceTransformer(self.embedding_model_name)

        texts = self._get_texts_for_indexing()

        # Генерируем эмбеддинги батчами
        print(f"Generating embeddings for {len(texts)} blocks...")
        embeddings = self.embedding_model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True  # Для косинусного сходства
        )

        self.dense_vectors = embeddings
        print(f"Dense index built: shape {self.dense_vectors.shape}")

    def save_index(self, output_dir: str):
        """Сохраняет все индексы в директорию"""
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        # Сохраняем корпус
        with open(output_path / "corpus_blocks.pkl", "wb") as f:
            pickle.dump(self.corpus_blocks, f)

        # Сохраняем BM25
        if self.bm25 is not None:
            with open(output_path / "bm25_index.pkl", "wb") as f:
                pickle.dump(self.bm25, f)

        # Сохраняем Dense векторы
        if self.dense_vectors is not None:
            np.save(output_path / "dense_vectors.npy", self.dense_vectors)

        # Сохраняем метаданные
        metadata = {
            "embedding_model": self.embedding_model_name,
            "num_blocks": len(self.corpus_blocks),
            "block_id_to_idx": self.block_id_to_idx
        }
        with open(output_path / "metadata.json", "w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

        print(f"Index saved to {output_dir}")


In [ ]:
# rag_system/retriever.py
import json
import pickle
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import heapq

class HybridRetriever:
    def __init__(self, index_dir: str):
        self.index_dir = Path(index_dir)
        self.corpus_blocks = []
        self.block_id_to_idx = {}
        self.bm25 = None
        self.dense_vectors = None
        self.embedding_model = None
        self.embedding_model_name = None

    def load_index(self):
        """Загружает все индексы из директории"""
        # Загружаем метаданные
        with open(self.index_dir / "metadata.json", "r", encoding="utf-8") as f:
            metadata = json.load(f)
            self.embedding_model_name = metadata["embedding_model"]
            self.block_id_to_idx = metadata["block_id_to_idx"]

        # Загружаем корпус
        with open(self.index_dir / "corpus_blocks.pkl", "rb") as f:
            self.corpus_blocks = pickle.load(f)

        # Загружаем BM25
        bm25_path = self.index_dir / "bm25_index.pkl"
        if bm25_path.exists():
            with open(bm25_path, "rb") as f:
                self.bm25 = pickle.load(f)

        # Загружаем Dense векторы
        dense_path = self.index_dir / "dense_vectors.npy"
        if dense_path.exists():
            self.dense_vectors = np.load(dense_path)

        print(f"Loaded index with {len(self.corpus_blocks)} blocks")

    def search_bm25(self, query: str, top_k: int = 20) -> List[Tuple[int, float]]:
        """Поиск по BM25"""
        if self.bm25 is None:
            return []

        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)

        # Получаем топ-k индексов с наибольшими скорами
        top_indices = heapq.nlargest(top_k, range(len(scores)), key=lambda i: scores[i])
        return [(idx, scores[idx]) for idx in top_indices]

    def search_dense(self, query: str, top_k: int = 20) -> List[Tuple[int, float]]:
        """Поиск по Dense векторам"""
        if self.dense_vectors is None or self.embedding_model is None:
            return []

        # Получаем эмбеддинг запроса
        query_embedding = self.embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]

        # Косинусное сходство (векторы уже нормализованы)
        scores = np.dot(self.dense_vectors, query_embedding)

        # Получаем топ-k
        top_indices = heapq.nlargest(top_k, range(len(scores)), key=lambda i: scores[i])
        return [(idx, scores[idx]) for idx in top_indices]

    def reciprocal_rank_fusion(self,
                               bm25_results: List[Tuple[int, float]],
                               dense_results: List[Tuple[int, float]],
                               k: int = 60) -> List[Tuple[int, float]]:
        """
        Reciprocal Rank Fusion (RRF) для объединения результатов
        RRF score = sum(1 / (k + rank))
        """
        rrf_scores = {}

        # BM25 ранги
        for rank, (idx, _) in enumerate(bm25_results):
            if idx not in rrf_scores:
                rrf_scores[idx] = 0.0
            rrf_scores[idx] += 1.0 / (k + rank + 1)

        # Dense ранги
        for rank, (idx, _) in enumerate(dense_results):
            if idx not in rrf_scores:
                rrf_scores[idx] = 0.0
            rrf_scores[idx] += 1.0 / (k + rank + 1)

        # Сортируем по RRF score
        sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_results

    def hybrid_search(self,
                      query: str,
                      top_k_bm25: int = 20,
                      top_k_dense: int = 20,
                      top_k_final: int = 20) -> List[Tuple[int, float]]:
        """
        Гибридный поиск: BM25 + Dense + RRF
        Возвращает список (индекс блока, гибридный скор)
        """
        bm25_results = self.search_bm25(query, top_k_bm25)
        dense_results = self.search_dense(query, top_k_dense)

        # Объединяем через RRF
        hybrid_results = self.reciprocal_rank_fusion(bm25_results, dense_results)

        return hybrid_results[:top_k_final]

    def get_block_by_idx(self, idx: int) -> Dict:
        """Получает блок по индексу"""
        return self.corpus_blocks[idx]

    def get_block_by_id(self, block_id: str) -> Dict:
        """Получает блок по block_id"""
        idx = self.block_id_to_idx.get(block_id)
        if idx is not None:
            return self.corpus_blocks[idx]
        return None

In [9]:
# rag_system/reranker.py
from typing import List, Tuple, Dict
from sentence_transformers import CrossEncoder

# class Reranker:
#     def __init__(self, model_name: str = "BAAI/bge-reranker-v2-m3"):
#         self.model_name = model_name
#         self.model = None

#     def load_model(self):
#         """Загружает модель реранкера"""
#         if self.model is None:
#             print(f"Loading reranker model: {self.model_name}")
#             self.model = CrossEncoder(self.model_name)

class Reranker:
    def __init__(self, model_name: str = "BAAI/bge-reranker-v2-m3", device: str = "cpu"):
        self.model_name = model_name
        self.device = device
        self.model = None

    def load_model(self):
        if self.model is None:
            print(f"Loading reranker model: {self.model_name} on {self.device}")
            self.model = CrossEncoder(
                self.model_name,
                device=self.device,  # Принудительно CPU
                automodel_args={"torch_dtype": "float32"}
            )

    def rerank(self,
               query: str,
               blocks: List[Dict],
               top_k: int = 3) -> List[Tuple[Dict, float]]:
        """
        Реранкает блоки по релевантности к запросу
        Возвращает список (блок, скор)
        """
        if self.model is None:
            self.load_model()

        # Создаем пары (query, text)
        pairs = [[query, block['text']] for block in blocks]

        # Получаем скоры от CrossEncoder
        scores = self.model.predict(pairs)

        # Сортируем по скору (убывание)
        block_scores = list(zip(blocks, scores))
        block_scores.sort(key=lambda x: x[1], reverse=True)

        return block_scores[:top_k]


In [8]:
# rag_system/utils.py
from typing import List, Dict
import tiktoken

def count_tokens(text: str, model_name: str = "gpt-3.5-turbo") -> int:
    """
    Подсчитывает количество токенов в тексте
    Используем tiktoken как аппроксимацию (Qwen токенизатор похож)
    """
    try:
        encoding = tiktoken.encoding_for_model(model_name)
        return len(encoding.encode(text))
    except:
        # Fallback: примерная оценка 1 токен ≈ 4 символа для русского
        return len(text) // 4

def get_adjacent_blocks(block_id: str,
                        block_id_to_idx: Dict[str, int],
                        corpus_blocks: List[Dict],
                        window: int = 1) -> List[Dict]:
    """
    Получает соседние блоки для расширения контекста
    window=1 вернет [prev, current, next]
    """
    if block_id not in block_id_to_idx:
        return []

    idx = block_id_to_idx[block_id]
    result = []

    # Добавляем блоки в окне
    for i in range(max(0, idx - window), min(len(corpus_blocks), idx + window + 1)):
        result.append(corpus_blocks[i])

    return result

def format_context(blocks: List[Dict], max_tokens: int = 2000) -> str:
    """
    Форматирует блоки в строку контекста для LLM
    Обрезает если превышает max_tokens
    """
    context_parts = []
    total_tokens = 0

    for block in blocks:
        block_text = f"[block_id: {block['block_id']}, page: {block['page']}]\n{block['text']}\n\n"
        block_tokens = count_tokens(block_text)

        if total_tokens + block_tokens > max_tokens:
            break

        context_parts.append(block_text)
        total_tokens += block_tokens

    return "".join(context_parts)

In [6]:
import json
with open('runtime.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

embedding_model = config.get("retrieval", {}).get("embedding_model", "BAAI/bge-m3")

# Создаем индексер
indexer = CorpusIndexer(embedding_model_name=embedding_model)

# Загружаем корпус
indexer.load_corpus('corpus.jsonl')

# Строим индексы
indexer.build_bm25_index()
indexer.build_dense_index(batch_size=32)

# Сохраняем
indexer.save_index('./index')
print("Index building completed successfully!")


Loaded 574 blocks
BM25 index built
Loading embedding model: BAAI/bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Generating embeddings for 574 blocks...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Dense index built: shape (574, 1024)
Index saved to ./index
Index building completed successfully!


In [10]:
import argparse
import json
import time
from pathlib import Path
from typing import List, Dict
import requests


class RAGPredictor:
    def __init__(self, index_dir: str, config_path: str):
        with open(config_path, 'r', encoding='utf-8') as f:
            self.config = json.load(f)

        # Загружаем ретривер
        self.retriever = HybridRetriever(index_dir)
        self.retriever.load_index()

        # Загружаем embedding модель для ретривера
        embedding_model_name = self.config["retrieval"]["embedding_model"]
        from sentence_transformers import SentenceTransformer
        self.retriever.embedding_model = SentenceTransformer(embedding_model_name)

        # Инициализируем реранкер
        reranker_model = self.config["retrieval"]["reranker_model"]
        self.reranker = Reranker(model_name=reranker_model)

        # Параметры из конфига
        self.retrieval_config = self.config["retrieval"]
        self.limits = self.config["limits"]
        self.gen_params = self.config["generation_params"]
        self.access_config = self.config["access"]

    def call_llm(self, messages: List[Dict], question_id: str) -> Dict:
        """
        Вызывает LLM через OpenAI-совместимый API
        Возвращает ответ и статистику по токенам
        """
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.access_config['api_key']}",
            "X-Question-ID": question_id,
            "X-Submission-ID": "SUBMISSION_ID"  # Замените на реальный ID
        }

        payload = {
            "model": self.access_config["model"],
            "messages": messages,
            "temperature": self.gen_params.get("temperature", 0.1),
            "top_p": self.gen_params.get("top_p", 0.9),
            "max_tokens": self.gen_params.get("max_tokens", 1024),
        }

        response = requests.post(
            f"{self.access_config['base_url']}/chat/completions",
            headers=headers,
            json=payload,
            timeout=self.limits.get("timeout_seconds", 60)
        )

        if response.status_code != 200:
            raise Exception(f"LLM API error: {response.status_code} - {response.text}")

        result = response.json()

        # Извлекаем статистику токенов
        usage = result.get("usage", {})
        input_tokens = usage.get("prompt_tokens", 0)
        output_tokens = usage.get("completion_tokens", 0)

        return {
            "content": result["choices"][0]["message"]["content"],
            "input_tokens": input_tokens,
            "output_tokens": output_tokens
        }

    def retrieve_context(self, query: str, question_id: str) -> List[Dict]:
        """
        Извлекает релевантный контекст для запроса
        """
        # Hybrid search
        hybrid_results = self.retriever.hybrid_search(
            query,
            top_k_bm25=self.retrieval_config.get("top_k_bm25", 20),
            top_k_dense=self.retrieval_config.get("top_k_dense", 20),
            top_k_final=20
        )

        # Получаем блоки
        candidate_blocks = [self.retriever.get_block_by_idx(idx) for idx, _ in hybrid_results]

        # Расширяем контекст соседними блоками
        if self.retrieval_config.get("use_adjacent_blocks", True):
            window = self.retrieval_config.get("adjacent_window", 1)
            expanded_blocks = []
            seen_ids = set()

            for block in candidate_blocks[:5]:  # Берем топ-5 для расширения
                adjacent = get_adjacent_blocks(
                    block['block_id'],
                    self.retriever.block_id_to_idx,
                    self.retriever.corpus_blocks,
                    window=window
                )
                for adj_block in adjacent:
                    if adj_block['block_id'] not in seen_ids:
                        expanded_blocks.append(adj_block)
                        seen_ids.add(adj_block['block_id'])

            candidate_blocks = expanded_blocks

        # Реранкинг
        top_k_rerank = self.retrieval_config.get("top_k_rerank", 3)
        reranked = self.reranker.rerank(query, candidate_blocks, top_k=top_k_rerank)

        return [block for block, _ in reranked]

    def generate_answer(self, question: Dict, context_blocks: List[Dict], question_id: str) -> Dict:
        """
        Генерирует ответ на вопрос с использованием контекста
        """
        # Форматируем контекст
        max_context_tokens = 2500  # Оставляем место для question + answer
        context = format_context(context_blocks, max_tokens=max_context_tokens)

        # Формируем промпт
        question_text = question["question"]
        options_text = "\n".join([f"- {opt['id']}. {opt['text']}" for opt in question["options"]])

        system_prompt = """You are a precise mathematical QA assistant. You must answer ONLY based on the provided context.

Rules:
1. Analyze the context carefully
2. Select ALL correct options (can be multiple)
3. Provide brief rationale (max 1200 chars)
4. Cite relevant block_ids (max 4)
5. If context is insufficient, set abstain=true

Output JSON format:
{
  "answer_ids": ["A", "C"],
  "abstain": false,
  "rationale": "Brief explanation",
  "citations": ["b_0042", "b_0107"]
}"""

        user_prompt = f"""Context:
{context}

Question:
{question_text}

Options:
{options_text}

Provide your answer in JSON format."""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        start_time = time.time()
        result = self.call_llm(messages, question_id)
        elapsed_ms = int((time.time() - start_time) * 1000)

        # Парсим JSON ответ
        try:
            # Извлекаем JSON из ответа (может быть в markdown блоке)
            content = result["content"]
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0].strip()
            elif "```" in content:
                content = content.split("```")[1].split("```")[0].strip()

            answer = json.loads(content)

            # Валидация
            if not isinstance(answer.get("answer_ids"), list):
                answer["answer_ids"] = []
            if not isinstance(answer.get("citations"), list):
                answer["citations"] = []
            if not isinstance(answer.get("abstain"), bool):
                answer["abstain"] = False

            # Ограничиваем citations до 4
            answer["citations"] = answer["citations"][:4]

        except Exception as e:
            # Если парсинг не удался, возвращаем пустой ответ
            answer = {
                "answer_ids": [],
                "abstain": True,
                "rationale": f"Failed to parse LLM response: {str(e)}",
                "citations": []
            }

        return {
            "answer": answer,
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "elapsed_ms": elapsed_ms,
            "retrieved_block_ids": [b["block_id"] for b in context_blocks]
        }

    def predict_question(self, question: Dict) -> Dict:
        """
        Делает предсказание для одного вопроса
        """
        question_id = question["id"]

        # Извлекаем контекст
        context_blocks = self.retrieve_context(question["question"], question_id)

        # Генерируем ответ
        result = self.generate_answer(question, context_blocks, question_id)

        # Формируем выходной формат
        output = {
            "id": question_id,
            "answer_ids": result["answer"]["answer_ids"],
            "abstain": result["answer"]["abstain"],
            "rationale": result["answer"]["rationale"],
            "citations": result["answer"]["citations"]
        }

        # Формируем trace
        trace = {
            "id": question_id,
            "llm_calls": 1,
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "elapsed_ms": result["elapsed_ms"],
            "retrieved_block_ids": result["retrieved_block_ids"]
        }

        return output, trace

    def predict_all(self, questions_path: str, output_path: str, trace_path: str):
        """
        Делает предсказания для всех вопросов
        """
        # Загружаем вопросы
        questions = []
        with open(questions_path, 'r', encoding='utf-8') as f:
            for line in f:
                questions.append(json.loads(line.strip()))

        print(f"Processing {len(questions)} questions...")

        # Открываем файлы для записи
        with open(output_path, 'w', encoding='utf-8') as f_out, \
                open(trace_path, 'w', encoding='utf-8') as f_trace:

            for i, question in enumerate(questions, 1):
                print(f"Processing question {i}/{len(questions)}: {question['id']}")

                try:
                    output, trace = self.predict_question(question)

                    # Записываем результаты
                    f_out.write(json.dumps(output, ensure_ascii=False) + "\n")
                    f_out.flush()

                    f_trace.write(json.dumps(trace, ensure_ascii=False) + "\n")
                    f_trace.flush()

                except Exception as e:
                    print(f"Error processing {question['id']}: {str(e)}")
                    # Записываем пустой ответ с ошибкой
                    error_output = {
                        "id": question["id"],
                        "answer_ids": [],
                        "abstain": True,
                        "rationale": f"Error: {str(e)}",
                        "citations": []
                    }
                    f_out.write(json.dumps(error_output, ensure_ascii=False) + "\n")
                    f_out.flush()

        print(f"Predictions saved to {output_path}")
        print(f"Trace saved to {trace_path}")


In [13]:
!tail -n 50 vllm.log

(EngineCore pid=31274) WARNING 11-05 22:24:55 [utils.py:333] Attempted to load weight blocks.7.mlp.linear_fc1.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=31274) WARNING 11-05 22:24:55 [utils.py:333] Attempted to load weight blocks.7.mlp.linear_fc2.bias with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=31274) WARNING 11-05 22:24:55 [utils.py:333] Attempted to load weight blocks.7.mlp.linear_fc2.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=31274) WARNING 11-05 22:24:55 [utils.py:333] Attempted to load weight blocks.7.norm1.bias with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=31274) WARNING 11-05 22:24:55 [utils.py:333] Attempted to load weight blocks.7.norm1.weight with dtype torch.bfloat16 into parameter with dtype torch.float16
(EngineCore pid=31274) WARNING 11-05 22:24:55 [utils.py:333] Attempted to load weight blocks.7.norm2.bias w

In [12]:
import time
import requests

print("Waiting for vLLM server...")
for i in range(60):
    try:
        response = requests.get("http://localhost:8000/health", timeout=2)
        if response.status_code == 200:
            print("✓ vLLM is ready!")

            # Тестовый запрос
            test_payload = {
                "model": "axteam/Qwen3.5-4B-AWQ-4bit",
                "messages": [{"role": "user", "content": "Hello"}],
                "max_tokens": 10
            }
            test_response = requests.post(
                "http://localhost:8000/v1/chat/completions",
                json=test_payload
            )
            print(f"Test response: {test_response.status_code}")
            break
    except Exception as e:
        if i % 10 == 0:
            print(f"Attempt {i}: {str(e)}")
    time.sleep(1)
else:
    print("✗ vLLM failed to start!")
    print("\n=== Last 30 lines of vllm.log ===")
    !tail -n 30 vllm.log

Waiting for vLLM server...
Attempt 0: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b5a6ffe7770>: Failed to establish a new connection: [Errno 111] Connection refused'))
Attempt 10: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b5a6ff01d00>: Failed to establish a new connection: [Errno 111] Connection refused'))
Attempt 20: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b5a6ff02360>: Failed to establish a new connection: [Errno 111] Connection refused'))


KeyboardInterrupt: 

In [31]:
predictor = RAGPredictor('index', 'runtime.json')

predictor.predict_all('dev_questions.jsonl', 'dev_gold.jsonl', 'trace.jsonl')

Loaded index with 574 blocks


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Processing 15 questions...
Processing question 1/15: q_001
Loading reranker model: BAAI/bge-reranker-v2-m3 on cuda


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Error processing q_001: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d92984dde10>: Failed to establish a new connection: [Errno 111] Connection refused'))
Processing question 2/15: q_002
Error processing q_002: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d90e1eb4490>: Failed to establish a new connection: [Errno 111] Connection refused'))
Processing question 3/15: q_003
Error processing q_003: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d90e1eb6140>: Failed to establish a new connection: [Errno 111] Connection refused'))
Processing question 4/15: q_004
Error processing q_004: HTTPConnec

KeyboardInterrupt: 